In [3]:
%pip install "psycopg[binary]"

   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.7 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.7 MB 699.0 kB/s eta 0:00:05
   ----- ---------------------------------- 0.5/3.7 MB 699.0 kB/s eta 0:00:05
   ----- ---------------------------------- 0.5/3.7 MB 699.0 kB/s eta 0:00:05
   -------- ------------------------------- 0.8/3.7 MB 610.3 kB/s eta 0:00:05
   -------- ------------------------------- 0.8/3.7 MB 610.3 kB/s eta 0:00:05
   ----------- ---------------------------- 1.0/3.7 MB 541.2 kB/s eta 0:00:05
   ----------- ---------------------------- 1.0/3.7 MB 541.2 kB/s eta 0:00:05
   ----------- ---------------------------- 1.0/3.7 MB 541.2 kB/s eta 0:00:05
   ----------- ---------------------------- 1.0/3.7 MB 541.2 kB/s eta 0:00:05
   ----------- ---------------------------- 1.0/3.7 MB 541.2 kB/s eta 0:00:05
   ----------

In [2]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

conn = psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
)

print("Подключение к PostgreSQL успешно!")
conn.close()

Подключение к PostgreSQL успешно!


In [4]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS trips (
                key TEXT PRIMARY KEY,
                fare_amount DOUBLE PRECISION,
                pickup_datetime TIMESTAMP,
                pickup_longitude DOUBLE PRECISION,
                pickup_latitude DOUBLE PRECISION,
                dropoff_longitude DOUBLE PRECISION,
                dropoff_latitude DOUBLE PRECISION,
                passenger_count INTEGER
            );
        """)

    print("Таблица trips успешно создана.")

Таблица trips успешно создана.


In [6]:
import duckdb

source_file = r"E:\laba_results\clean_final.parquet"

con = duckdb.connect()

schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [source_file]).fetchdf()

display(schema)

con.close()

,column_name,column_type,null,key,default,extra
0,key,VARCHAR,YES,None,None,None
1,fare_amount,DOUBLE,YES,None,None,None
2,pickup_datetime,TIMESTAMP,YES,None,None,None
3,pickup_longitude,DOUBLE,YES,None,None,None
4,pickup_latitude,DOUBLE,YES,None,None,None
5,dropoff_longitude,DOUBLE,YES,None,None,None
6,dropoff_latitude,DOUBLE,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None


In [7]:
import duckdb
import psycopg
from getpass import getpass

source_file = r"E:\laba_results\clean_final.parquet"
password = getpass("Введите пароль PostgreSQL: ")

# Читаем только 5 строк из Parquet
duck = duckdb.connect()
rows = duck.execute("""
    SELECT *
    FROM read_parquet(?)
    LIMIT 5
""", [source_file]).fetchall()
duck.close()

# Пробная запись в PostgreSQL
with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO trips (
                key, fare_amount, pickup_datetime,
                pickup_longitude, pickup_latitude,
                dropoff_longitude, dropoff_latitude,
                passenger_count
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (key) DO NOTHING
        """, rows)

        cur.execute("SELECT COUNT(*) FROM trips")
        print("Строк в таблице после теста:", cur.fetchone()[0])

print("Пробная запись завершена.")

Строк в таблице после теста: 5
Пробная запись завершена.


In [8]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE trips")
        cur.execute("SELECT COUNT(*) FROM trips")
        print("Строк после очистки:", cur.fetchone()[0])

Строк после очистки: 0


In [10]:
import time
import duckdb
import psycopg
from getpass import getpass

source_file = r"E:\laba_results\clean_final.parquet"
batch_size = 10_000
password = getpass("Введите пароль PostgreSQL: ")

insert_sql = """
    INSERT INTO trips (
        key, fare_amount, pickup_datetime,
        pickup_longitude, pickup_latitude,
        dropoff_longitude, dropoff_latitude,
        passenger_count
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (key) DO NOTHING
"""

start_time = time.perf_counter()
total_inserted = 0
total_read = 0

duck = duckdb.connect()
reader = duck.execute("""
    SELECT *
    FROM read_parquet(?)
""", [source_file])

try:
    with psycopg.connect(
        host="127.0.0.1",
        port=5432,
        dbname="taxi_db",
        user="postgres",
        password=password,
        connect_timeout=10
    ) as conn:
        with conn.cursor() as cur:
            while True:
                rows = reader.fetchmany(batch_size)
                if not rows:
                    break

                total_read += len(rows)
                cur.executemany(insert_sql, rows)
                total_inserted += len(rows)

                if total_read % 500_000 < batch_size:
                    elapsed = time.perf_counter() - start_time
                    print(
                        f"Прочитано: {total_read:,}; "
                        f"время: {elapsed / 60:.1f} мин."
                    )

    print("\nЗагрузка завершена.")
    print(f"Прочитано строк: {total_read:,}")
    print(f"Время загрузки: {time.perf_counter() - start_time:.2f} сек.")

finally:
    duck.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Прочитано: 500,000; время: 0.8 мин.
Прочитано: 1,000,000; время: 1.9 мин.
Прочитано: 1,500,000; время: 3.0 мин.
Прочитано: 2,000,000; время: 4.2 мин.
Прочитано: 2,500,000; время: 5.3 мин.
Прочитано: 3,000,000; время: 6.5 мин.
Прочитано: 3,500,000; время: 7.6 мин.
Прочитано: 4,000,000; время: 8.5 мин.
Прочитано: 4,500,000; время: 9.6 мин.
Прочитано: 5,000,000; время: 10.8 мин.
Прочитано: 5,500,000; время: 12.1 мин.
Прочитано: 6,000,000; время: 13.3 мин.
Прочитано: 6,500,000; время: 14.5 мин.
Прочитано: 7,000,000; время: 15.8 мин.
Прочитано: 7,500,000; время: 17.0 мин.
Прочитано: 8,000,000; время: 18.2 мин.
Прочитано: 8,500,000; время: 19.4 мин.
Прочитано: 9,000,000; время: 20.6 мин.
Прочитано: 9,500,000; время: 21.9 мин.
Прочитано: 10,000,000; время: 23.1 мин.
Прочитано: 10,500,000; время: 24.4 мин.
Прочитано: 11,000,000; время: 25.7 мин.
Прочитано: 11,500,000; время: 26.9 мин.
Прочитано: 12,000,000; время: 28.2 мин.
Прочитано: 12,500,000; время: 29.5 мин.
Прочитано: 13,000,000; время: 

In [11]:
import time
import psycopg
import pandas as pd
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

conn = psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
)

results = []

def measure_query(name, sql, params=None, repeats=3):
    """Измеряет время SQL-запроса и сохраняет лучший результат."""
    times = []

    with conn.cursor() as cur:
        for _ in range(repeats):
            start = time.perf_counter()
            cur.execute(sql, params or ())
            cur.fetchall()
            times.append(time.perf_counter() - start)

    best = min(times)
    average = sum(times) / len(times)

    results.append({
        "Показатель": name,
        "Лучшее время, сек": round(best, 4),
        "Среднее время, сек": round(average, 4)
    })

    print(
        f"{name}: лучшее {best:.4f} сек., "
        f"среднее {average:.4f} сек."
    )

try:
    # 1. Размер базы данных
    with conn.cursor() as cur:
        cur.execute("""
            SELECT pg_database_size(current_database())
        """)
        size_bytes = cur.fetchone()[0]

    size_mb = size_bytes / (1024 ** 2)
    results.append({
        "Показатель": "Размер базы данных, МБ",
        "Лучшее время, сек": None,
        "Среднее время, сек": round(size_mb, 2)
    })

    print(f"Размер базы taxi_db: {size_mb:.2f} МБ")

    # 2. Агрегирующий запрос
    measure_query(
        "Агрегация стоимости поездок",
        """
        SELECT
            passenger_count,
            COUNT(*) AS trip_count,
            AVG(fare_amount) AS average_fare,
            SUM(fare_amount) AS total_fare
        FROM trips
        GROUP BY passenger_count
        ORDER BY passenger_count
        """
    )

    # 3. Запрос по 2010 году без индекса
    with conn.cursor() as cur:
        cur.execute("""
            DROP INDEX IF EXISTS idx_trips_pickup_datetime
        """)
    conn.commit()

    date_query = """
        SELECT COUNT(*)
        FROM trips
        WHERE pickup_datetime >= %s
          AND pickup_datetime < %s
    """

    measure_query(
        "Запрос за 2010 год без индекса",
        date_query,
        ("2010-01-01", "2011-01-01")
    )

    # 4. Создание индекса
    start = time.perf_counter()

    with conn.cursor() as cur:
        cur.execute("""
            CREATE INDEX idx_trips_pickup_datetime
            ON trips (pickup_datetime)
        """)
    conn.commit()

    index_time = time.perf_counter() - start

    results.append({
        "Показатель": "Создание индекса, сек",
        "Лучшее время, сек": round(index_time, 4),
        "Среднее время, сек": None
    })

    print(f"Создание индекса: {index_time:.4f} сек.")

    # 5. Тот же запрос после создания индекса
    measure_query(
        "Запрос за 2010 год с индексом",
        date_query,
        ("2010-01-01", "2011-01-01")
    )

    # Итоговая таблица
    display(pd.DataFrame(results))

finally:
    conn.close()

Размер базы taxi_db: 9621.44 МБ
Агрегация стоимости поездок: лучшее 8.6525 сек., среднее 29.8011 сек.
Запрос за 2010 год без индекса: лучшее 5.4341 сек., среднее 5.6862 сек.
Создание индекса: 53.4441 сек.
Запрос за 2010 год с индексом: лучшее 9.3916 сек., среднее 59.1903 сек.


,Показатель,"Лучшее время, сек","Среднее время, сек"
0,"Размер базы данных, МБ",NaN,9621.4400
1,Агрегация стоимости поездок,8.6525,29.8011
2,Запрос за 2010 год без индекса,5.4341,5.6862
3,"Создание индекса, сек",53.4441,NaN
4,Запрос за 2010 год с индексом,9.3916,59.1903


In [13]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            EXPLAIN
            SELECT COUNT(*)
            FROM trips
            WHERE pickup_datetime >= '2010-01-01'
              AND pickup_datetime < '2011-01-01';
        """)

        for row in cur.fetchall():
            print(row[0])

Finalize Aggregate  (cost=149744.94..149744.95 rows=1 width=8)
  ->  Gather  (cost=149744.72..149744.93 rows=2 width=8)
        Workers Planned: 2
        ->  Partial Aggregate  (cost=148744.72..148744.73 rows=1 width=8)
              ->  Parallel Index Only Scan using idx_trips_pickup_datetime on trips  (cost=0.56..140136.90 rows=3443128 width=0)
                    Index Cond: ((pickup_datetime >= '2010-01-01 00:00:00'::timestamp without time zone) AND (pickup_datetime < '2011-01-01 00:00:00'::timestamp without time zone))


In [14]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            EXPLAIN (ANALYZE, BUFFERS)
            SELECT COUNT(*)
            FROM trips
            WHERE pickup_datetime >= '2010-01-01'
              AND pickup_datetime < '2011-01-01';
        """)

        for row in cur.fetchall():
            print(row[0])

Finalize Aggregate  (cost=149744.94..149744.95 rows=1 width=8) (actual time=930.599..947.348 rows=1.00 loops=1)
  Buffers: shared hit=6800988 read=14990
  ->  Gather  (cost=149744.72..149744.93 rows=2 width=8) (actual time=930.303..947.342 rows=3.00 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        Buffers: shared hit=6800988 read=14990
        ->  Partial Aggregate  (cost=148744.72..148744.73 rows=1 width=8) (actual time=845.963..845.963 rows=1.00 loops=3)
              Buffers: shared hit=6800988 read=14990
              ->  Parallel Index Only Scan using idx_trips_pickup_datetime on trips  (cost=0.56..140136.90 rows=3443128 width=0) (actual time=0.314..736.872 rows=2784882.67 loops=3)
                    Index Cond: ((pickup_datetime >= '2010-01-01 00:00:00'::timestamp without time zone) AND (pickup_datetime < '2011-01-01 00:00:00'::timestamp without time zone))
                    Heap Fetches: 0
                    Index Searches: 1
                    Buffer

In [15]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
              AND table_type = 'BASE TABLE'
            ORDER BY table_name;
        """)
        print("Таблицы:")
        for row in cur.fetchall():
            print("-", row[0])

        cur.execute("""
            SELECT indexname
            FROM pg_indexes
            WHERE schemaname = 'public'
              AND tablename = 'trips'
            ORDER BY indexname;
        """)
        print("\nИндексы таблицы trips:")
        for row in cur.fetchall():
            print("-", row[0])

Таблицы:
- trips

Индексы таблицы trips:
- idx_trips_pickup_datetime
- trips_pkey


In [16]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS passenger_categories (
                passenger_count INTEGER PRIMARY KEY,
                category_name TEXT NOT NULL
            );
        """)

        cur.execute("""
            INSERT INTO passenger_categories
                (passenger_count, category_name)
            VALUES
                (0, '0 пассажиров'),
                (1, '1 пассажир'),
                (2, '2 пассажира'),
                (3, '3 пассажира'),
                (4, '4 пассажира'),
                (5, '5 пассажиров'),
                (6, '6 пассажиров'),
                (7, '7 пассажиров'),
                (8, '8 пассажиров'),
                (9, '9 пассажиров'),
                (10, '10 пассажиров'),
                (11, '11 пассажиров'),
                (12, '12 пассажиров'),
                (13, '13 пассажиров')
            ON CONFLICT (passenger_count) DO NOTHING;
        """)

        cur.execute("SELECT COUNT(*) FROM passenger_categories")
        print("Категорий в справочнике:", cur.fetchone()[0])

Категорий в справочнике: 14


In [17]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT DISTINCT t.passenger_count
            FROM trips AS t
            LEFT JOIN passenger_categories AS p
                ON t.passenger_count = p.passenger_count
            WHERE p.passenger_count IS NULL
            ORDER BY t.passenger_count;
        """)

        missing = cur.fetchall()

        if missing:
            print("Категории, отсутствующие в справочнике:")
            for row in missing:
                print(row[0])
        else:
            print("Все категории поездок представлены в справочнике.")

Категории, отсутствующие в справочнике:
34
49
129
208


In [18]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO passenger_categories
                (passenger_count, category_name)
            VALUES
                (34, '34 пассажира'),
                (49, '49 пассажиров'),
                (129, '129 пассажиров'),
                (208, '208 пассажиров')
            ON CONFLICT (passenger_count) DO NOTHING;
        """)

        cur.execute("""
            SELECT COUNT(*)
            FROM passenger_categories;
        """)

        print("Всего категорий в справочнике:", cur.fetchone()[0])

Всего категорий в справочнике: 18


In [19]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT COUNT(*)
            FROM (
                SELECT DISTINCT t.passenger_count
                FROM trips AS t
                LEFT JOIN passenger_categories AS p
                    ON t.passenger_count = p.passenger_count
                WHERE p.passenger_count IS NULL
            ) AS missing;
        """)

        missing_count = cur.fetchone()[0]

        print("Категорий, отсутствующих в справочнике:", missing_count)

Категорий, отсутствующих в справочнике: 0


In [20]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            ALTER TABLE trips
            ADD CONSTRAINT fk_trips_passenger_category
            FOREIGN KEY (passenger_count)
            REFERENCES passenger_categories (passenger_count);
        """)

    print("Внешний ключ успешно добавлен.")

Внешний ключ успешно добавлен.


In [21]:
import psycopg
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                conname,
                contype
            FROM pg_constraint
            WHERE conrelid = 'public.trips'::regclass
            ORDER BY conname;
        """)

        print("Ограничения таблицы trips:")
        for name, constraint_type in cur.fetchall():
            types = {
                "p": "PRIMARY KEY",
                "f": "FOREIGN KEY",
                "u": "UNIQUE",
                "c": "CHECK"
            }
            print(f"- {name}: {types.get(constraint_type, constraint_type)}")

        cur.execute("""
            SELECT indexname
            FROM pg_indexes
            WHERE schemaname = 'public'
              AND tablename = 'trips'
            ORDER BY indexname;
        """)

        print("\nИндексы таблицы trips:")
        for (index_name,) in cur.fetchall():
            print("-", index_name)

Ограничения таблицы trips:
- fk_trips_passenger_category: FOREIGN KEY
- trips_key_not_null: n
- trips_pkey: PRIMARY KEY

Индексы таблицы trips:
- idx_trips_pickup_datetime
- trips_pkey


In [22]:
import time
import statistics
import psycopg
import pandas as pd
from getpass import getpass

password = getpass("Введите пароль PostgreSQL: ")

queries = {
    "Агрегация стоимости поездок": """
        SELECT SUM(fare_amount)
        FROM trips;
    """,
    "Запрос за 2010 год без индекса": """
        SELECT COUNT(*)
        FROM trips
        WHERE pickup_datetime >= TIMESTAMP '2010-01-01'
          AND pickup_datetime < TIMESTAMP '2011-01-01';
    """,
    "Запрос за 2010 год с индексом": """
        SELECT COUNT(*)
        FROM trips
        WHERE pickup_datetime >= TIMESTAMP '2010-01-01'
          AND pickup_datetime < TIMESTAMP '2011-01-01';
    """
}

results = []

with psycopg.connect(
    host="127.0.0.1",
    port=5432,
    dbname="taxi_db",
    user="postgres",
    password=password,
    connect_timeout=10
) as conn:
    with conn.cursor() as cur:
        # Проверяем, что индекс существует
        cur.execute("""
            SELECT EXISTS (
                SELECT 1
                FROM pg_indexes
                WHERE schemaname = 'public'
                  AND tablename = 'trips'
                  AND indexname = 'idx_trips_pickup_datetime'
            );
        """)
        index_exists = cur.fetchone()[0]
        print("Индекс по дате существует:", index_exists)

        for name, query in queries.items():
            # Три измерения одного и того же запроса.
            # Для второго теста план принудительно выбирает
            # последовательное чтение таблицы.
            times = []

            for run in range(3):
                if name == "Запрос за 2010 год без индекса":
                    cur.execute("SET enable_indexscan = off;")
                    cur.execute("SET enable_indexonlyscan = off;")
                    cur.execute("SET enable_bitmapscan = off;")
                else:
                    cur.execute("SET enable_indexscan = on;")
                    cur.execute("SET enable_indexonlyscan = on;")
                    cur.execute("SET enable_bitmapscan = on;")

                start = time.perf_counter()
                cur.execute(query)
                value = cur.fetchone()[0]
                times.append(time.perf_counter() - start)

            # Возвращаем стандартные настройки планировщика
            cur.execute("RESET enable_indexscan;")
            cur.execute("RESET enable_indexonlyscan;")
            cur.execute("RESET enable_bitmapscan;")

            results.append({
                "Запрос": name,
                "Результат": value,
                "Лучшее время, сек.": round(min(times), 4),
                "Среднее время, сек.": round(statistics.mean(times), 4),
                "Все замеры, сек.": [round(t, 4) for t in times]
            })

display(pd.DataFrame(results))

Индекс по дате существует: True


,Запрос,Результат,"Лучшее время, сек.","Среднее время, сек.","Все замеры, сек."
0,Агрегация стоимости поездок,6.285262e+08,6.6415,7.7079,"[6.6415, 8.292, 8.1902]"
1,Запрос за 2010 год без индекса,8.354648e+06,7.8605,8.1204,"[8.1521, 7.8605, 8.3485]"
2,Запрос за 2010 год с индексом,8.354648e+06,0.6192,0.7555,"[0.99, 0.6571, 0.6192]"
